In [1]:
import diffrax
import equinox as eqx
import jax.numpy as jnp
import jax

In [2]:
class Engine(eqx.Module):
    temperature: jax.Array
    
    def get_friction(self):
        # Метод класса, используемый в расчетах
        return self.temperature * 0.1

class Vehicle(eqx.Module):
    position: jax.Array
    velocity: jax.Array
    engine: Engine  # Вложенный класс

# 1. Определяем векторное поле как функцию, принимающую структуру
def vector_field(t, y, args):
    # y здесь — это экземпляр Vehicle! 
    # Мы можем вызывать его методы и обращаться к полям
    friction = y.engine.get_friction()
    
    d_pos = y.velocity
    d_vel = -friction * y.velocity
    d_temp = jnp.array(0.05) # мотор греется
    
    # Возвращаем структуру той же формы, что и y
    return Vehicle(position=d_pos, velocity=d_vel, engine=Engine(d_temp))

# 2. Начальное состояние — полноценный объект
y0 = Vehicle(
    position=jnp.array(0.0), 
    velocity=jnp.array(10.0), 
    engine=Engine(temperature=jnp.array(20.0))
)

In [3]:

# 3. Решаем уравнение
term = diffrax.ODETerm(vector_field)
solver = diffrax.Tsit5()
sol = diffrax.diffeqsolve(term, solver, t0=0, t1=10, dt0=0.1, y0=y0)

# Результат sol.ys тоже будет структурой объектов Vehicle!
final_vehicle = jax.tree_util.tree_map(lambda x: x[-1], sol.ys)
print(f"Final temp: {final_vehicle.engine.temperature}")

Final temp: 20.499916076660156
